In [13]:
# ============================================================
# IMPORTS
# ============================================================
import os
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
import seaborn as sns
from datetime import datetime
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

# ============================================================
# REPRODUCIBILITY
# ============================================================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================
# SECTION 0: CONFIGURATION
# ============================================================
CONFIG = {
    # Data
    'ticker':           'SPY',
    'start_date':       '2000-01-01',
    # Chronological splits
    'train_end':        '2018-01-01',
    'val_end':          '2023-01-01',
    # Feature / sequence settings
    'lookback':         30,
    'forecast_horizon': 5,
    'features': [
        'rv_daily', 'rv_weekly', 'rv_monthly',  # HAR components
        'log_return',                            # price momentum proxy
        'parkinson_vol',                         # high-low estimator
        'rsi_14',                                # market microstructure
        'rel_volume',                            # liquidity proxy
        'intraday_range',                        # intraday volatility proxy
    ],
    # LSTM architecture
    'hidden1':    64,
    'hidden2':    32,
    'dropout':    0.3,
    # Training
    'batch_size': 32,
    'lr':         1e-3,
    'weight_decay': 1e-5,
    'max_epochs': 200,
    'patience':   15,
    'grad_clip':  1.0,
    # Backtest
    'cost_bps':   0,#5,   # one-way transaction cost in basis points
    # Outputs
    'model_path':   'lstm_vol_model.pt',
    'scaler_path':  'scaler.pkl',
    'metrics_path': 'metrics.csv',
    'report_path':  'report.md',
    'config_path':  'model_config.json',
    'plot_path':    'lstm_vol_report.png',
}


In [14]:
# ============================================================
# SECTION 1: DATA ACQUISITION
# ============================================================

def download_data(ticker: str, start: str) -> pd.DataFrame:
    """
    Download OHLCV data from yfinance. auto_adjust=True handles
    dividend and split adjustments.

    Returns a clean DataFrame with columns: Open, High, Low, Close, Volume.
    All prices are adjusted (split- and dividend-corrected).
    """
    print(f"\n[DATA] Downloading {ticker} from {start}…")
    raw = yf.download(ticker, start=start, auto_adjust=True, progress=False)

    # Flatten MultiIndex columns if present
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)

    raw = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
    raw.index = pd.to_datetime(raw.index)
    raw = raw.sort_index()

    # Validate: missing values
    n_nan = raw.isnull().sum().sum()
    if n_nan > 0:
        print(f"  WARNING: {n_nan} NaN values. Forward-filling.")
        raw = raw.ffill().dropna()

    # Validate: non-positive prices (data error)
    if (raw[['Open', 'High', 'Low', 'Close']] <= 0).any().any():
        raise ValueError("Non-positive price detected — data integrity error.")

    # Flag extreme moves (>15%) for awareness, but retain (COVID etc.)
    log_ret = np.log(raw['Close'] / raw['Close'].shift(1)).dropna()
    n_extreme = (np.abs(log_ret) > 0.15).sum()
    if n_extreme > 0:
        print(f"  INFO: {n_extreme} daily moves >15% flagged (retained).")

    print(f"  Shape: {raw.shape} | "
          f"{raw.index[0].date()} → {raw.index[-1].date()}")
    return raw

raw = download_data(CONFIG['ticker'], CONFIG['start_date'])



[DATA] Downloading SPY from 2000-01-01…
  Shape: (6633, 5) | 2000-01-03 → 2026-05-18


All features are strictly causal: at time t, only data from [0, t] is used.

Mathematical definitions
------------------------

r_t = log-return
$$
r_t = \log\frac{C_t}{C_{t-1}}
$$

Daily / weekly / monthly RV (HAR components)
$$
\mathrm{RV}_t^{d} = r_t^2
$$
$$
\mathrm{RV}_t^{w} = \frac{1}{5}\sum_{i=1}^{5} r_{t-i}^2
$$
$$
\mathrm{RV}_t^{m} = \frac{1}{22}\sum_{i=1}^{22} r_{t-i}^2
$$

Parkinson (1980) [C2 — corrected]:
$$
\sigma_P^2 = \frac{1}{4\ln 2}\,\mathbb{E}\!\left[\left(\ln\frac{H}{L}\right)^2\right]
$$
$$
\sigma_P = \sqrt{\frac{1}{4\ln 2}\cdot\mathrm{rolling\_mean}\!\left(\left(\ln\frac{H}{L}\right)^2,n\right)}
$$
NOTE: The spec erroneously wrote mean(log(H/L)) / (4 ln 2), which is dimensionally and statistically incorrect.

Garman–Klass (1980):
$$
\sigma_{GK}^2 = \tfrac{1}{2}\left(\ln\frac{H}{L}\right)^2 - (2\ln 2 - 1)\left(\ln\frac{C}{O}\right)^2
$$

RSI (Wilder, normalized to [0,1]):
$$
\mathrm{RSI} = 100 - \frac{100}{1 + RS},\qquad
RS = \frac{\mathrm{EMA}(\text{gains})}{\mathrm{EMA}(\text{losses})}
$$
$$
\mathrm{rsi\_14\_norm} = \frac{\mathrm{RSI}}{100}
$$

Other features
$$
\mathrm{rel\_volume}_t = \frac{V_t}{\mathrm{MA}_{20}(V_t)}
$$
$$
\mathrm{intraday\_range}_t = \frac{H_t - L_t}{C_{t-1}}
$$

In [15]:
# ============================================================
# SECTION 2: FEATURE ENGINEERING
# ============================================================

def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    All features are strictly causal: at time t, only data from [0, t] is used.

    Mathematical definitions
    ------------------------
    r_t          = log(C_t / C_{t-1})                           log-return
    RV_t^d       = r_t^2                                        daily RV proxy
    RV_t^w       = (1/5)  * sum_{i=1}^{5}  r_{t-i}^2          weekly HAR component
    RV_t^m       = (1/22) * sum_{i=1}^{22} r_{t-i}^2          monthly HAR component

    Parkinson (1980) [C2 — corrected]:
        sigma_P^2 = (1 / (4 ln 2)) * E[(log(H/L))^2]
        => sigma_P = sqrt( (1/(4 ln 2)) * rolling_mean( (log(H/L))^2, n ) )
        NOTE: The spec erroneously wrote mean(log(H/L)) / (4 ln 2), which
              is dimensionally and statistically incorrect.

    Garman-Klass (1980):
        sigma_GK^2 = 0.5*(log(H/L))^2 - (2 ln 2 - 1)*(log(C/O))^2

    RSI (Wilder, normalized to [0,1]):
        RSI = 100 - 100/(1 + RS),  RS = EMA(gains)/EMA(losses)
        rsi_14_norm = RSI / 100

    rel_volume   = V_t / MA20(V_t)
    intraday_range = (H_t - L_t) / C_{t-1}   (normalized by prior close)
    """
    feat = pd.DataFrame(index=df.index)

    # --- Log return ---
    feat['log_return'] = np.log(df['Close'] / df['Close'].shift(1))

    # --- HAR-RV components (Corsi 2009) ---
    rv_sq = feat['log_return'] ** 2
    feat['rv_daily']   = rv_sq
    feat['rv_weekly']  = rv_sq.rolling(5).mean()
    feat['rv_monthly'] = rv_sq.rolling(22).mean()

    # Rolling close-to-close vol (annualized, 20-day)
    feat['vol_20'] = feat['log_return'].rolling(20).std() * np.sqrt(252)

    # --- Parkinson (1980) high-low estimator [C2: corrected] ---
    hl_log_sq = np.log(df['High'] / df['Low']) ** 2   # square first
    feat['parkinson_vol'] = np.sqrt(
        hl_log_sq.rolling(20).mean() / (4.0 * np.log(2))
    )

    # --- Garman-Klass (1980) estimator ---
    hl2 = 0.5 * np.log(df['High'] / df['Low']) ** 2
    co2 = (2.0 * np.log(2) - 1.0) * np.log(df['Close'] / df['Open']) ** 2
    feat['gk_vol'] = np.sqrt((hl2 - co2).clip(lower=0).rolling(20).mean())

    # --- RSI (14-day, Wilder smoothing approximated by rolling mean) ---
    delta = df['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / (loss + 1e-10)
    feat['rsi_14'] = (100.0 - 100.0 / (1.0 + rs)) / 100.0   # normalized to [0,1]

    # --- Volume features ---
    vol_ma20 = df['Volume'].rolling(20).mean()
    feat['rel_volume']   = df['Volume'] / (vol_ma20 + 1e-10)
    feat['volume_trend'] = (df['Volume'].rolling(5).mean() /
                            (vol_ma20 + 1e-10))

    # --- Intraday range (normalized, causal: use prior close) ---
    feat['intraday_range'] = (df['High'] - df['Low']) / df['Close'].shift(1)

    # --- Close-to-open gap ---
    feat['gap'] = np.abs(df['Open'] - df['Close'].shift(1)) / df['Close'].shift(1)

    # --- Rolling skewness (5-day) ---
    feat['skew_5']    = feat['log_return'].rolling(5).skew()
    feat['mean_ret_5'] = feat['log_return'].rolling(5).mean()

    return feat
def compute_target(df: pd.DataFrame, horizon: int = 5) -> pd.Series:
    """
    5-day forward annualized realized volatility. [C1: corrected normalization]

    Target at time t:
        sigma_{t:t+H} = sqrt( (252/H) * sum_{i=1}^{H} r_{t+i}^2 )

    This is FORWARD-LOOKING (uses future returns) and only for target construction.
    The shift(-horizon) alignment ensures no look-ahead in features.

    Note: target[t] uses returns from t+1 to t+H (rolling sum shifted back H days).
    When aligned with sequences of lookback=L, the last feature day is t=i+L-1,
    so the effective prediction is vol from day i+L to i+L+H-1.
    """
    log_ret = np.log(df['Close'] / df['Close'].shift(1))
    rv_sq   = log_ret ** 2
    # Sum of next H squared returns, annualized
    target  = np.sqrt((252.0 / horizon) * rv_sq.rolling(horizon).sum().shift(-horizon))
    return target.rename('target_vol')

feat   = compute_features(raw)
target = compute_target(raw, CONFIG['forecast_horizon'])

data = pd.concat([feat, target], axis=1).dropna()
print(f"\n[DATASET] {len(data)} clean samples after dropna.")



[DATASET] 6606 clean samples after dropna.


In [16]:
# ============================================================
# SECTION 3: HAR-RV BASELINE  [C4 — added]
# ============================================================

class HARRVModel:
    """
    Heterogeneous Autoregressive Realized Volatility model (Corsi 2009).

    Specification:
        RV_t = beta_0 + beta_d * RV_{t-1}^{(d)}
                      + beta_w * RV_{t-1}^{(w)}
                      + beta_m * RV_{t-1}^{(m)}  + epsilon_t

    where RV_{t-1}^{(d)}, RV_{t-1}^{(w)}, RV_{t-1}^{(m)} are the daily,
    5-day-average, and 22-day-average realized variance components.

    Estimated by OLS. Consistently competitive with ML models on daily data;
    serves as the primary benchmark for DM-test comparison.
    """
    HAR_COLS = ['rv_daily', 'rv_weekly', 'rv_monthly']

    def __init__(self):
        self.model_   = None
        self.ols_res_ = None

    def fit(self, X_train: pd.DataFrame, y_train: np.ndarray) -> 'HARRVModel':
        X_ols = sm.add_constant(X_train[self.HAR_COLS].values)
        self.ols_res_ = sm.OLS(y_train, X_ols).fit(
            cov_type='HAC', cov_kwds={'maxlags': 10})
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        X_ols = sm.add_constant(X[self.HAR_COLS].values)
        return np.clip(self.ols_res_.predict(X_ols), a_min=0, a_max=None)

    def summary(self) -> str:
        return str(self.ols_res_.summary())

tr_mask  = data.index <= CONFIG['train_end']
vl_mask  = (data.index >  CONFIG['train_end']) & (data.index <= CONFIG['val_end'])
te_mask  = data.index >  CONFIG['val_end']

fcols = CONFIG['features']

X_tr = data.loc[tr_mask, fcols].values.astype(np.float32)
X_vl = data.loc[vl_mask, fcols].values.astype(np.float32)
X_te = data.loc[te_mask, fcols].values.astype(np.float32)
y_tr = data.loc[tr_mask, 'target_vol'].values.astype(np.float32)
y_vl = data.loc[vl_mask, 'target_vol'].values.astype(np.float32)
y_te = data.loc[te_mask, 'target_vol'].values.astype(np.float32)

te_dates = data.index[te_mask]
print(f"[SPLITS] Train: {len(X_tr)} | Val: {len(X_vl)} | Test: {len(X_te)}")


[SPLITS] Train: 4506 | Val: 1259 | Test: 841


In [17]:
# ============================================================
# SECTION 4: LSTM MODEL  [C3, C7 — corrected]
# ============================================================

class VolatilityLSTM(nn.Module):
    """
    Two-layer stacked LSTM for volatility forecasting.

    Architecture
    ------------
    Input  → [batch, lookback, n_features]
    LSTM-1 → hidden1 (no built-in dropout; single layer)
    Dropout(p)                              ← explicit [C3]
    LSTM-2 → hidden2
    Dropout(p)
    Linear(hidden2, 16) + ReLU
    Linear(16, 1) + Softplus             ← guarantees sigma > 0

    Softplus: f(x) = log(1 + e^x)
    Guarantees positivity while remaining differentiable at 0.
    Unlike ReLU, Softplus has no dead neuron problem for regression targets.

    Weight Initialization [C7]
    --------------------------
    - Input weights (W_ih): Xavier uniform
    - Recurrent weights (W_hh): Orthogonal (preserves gradient norms)
    - Biases: zero-initialized except forget gate bias = 1.0
      (Jozefowicz et al. 2015: reduces vanishing gradient at initialization)
    - Linear layers: Xavier uniform
    """

    def __init__(self, n_features: int, hidden1: int = 64, hidden2: int = 32,
                 dropout: float = 0.3):
        super().__init__()
        self.lstm1    = nn.LSTM(n_features, hidden1, batch_first=True, num_layers=1)
        self.drop1    = nn.Dropout(p=dropout)            # [C3] explicit dropout
        self.lstm2    = nn.LSTM(hidden1,    hidden2, batch_first=True, num_layers=1)
        self.drop2    = nn.Dropout(p=dropout)
        self.fc1      = nn.Linear(hidden2, 16)
        self.relu     = nn.ReLU()
        self.fc2      = nn.Linear(16, 1)
        self.softplus = nn.Softplus()

        self._init_weights()

    def _init_weights(self):  # [C7]
        for lstm in (self.lstm1, self.lstm2):
            for name, param in lstm.named_parameters():
                if 'weight_ih' in name:
                    nn.init.xavier_uniform_(param)
                elif 'weight_hh' in name:
                    nn.init.orthogonal_(param)
                elif 'bias' in name:
                    nn.init.zeros_(param)
                    # Set forget gate bias to 1 (units n//4 : n//2 in 4-gate layout)
                    n = param.size(0)
                    param.data[n // 4 : n // 2].fill_(1.0)
        for fc in (self.fc1, self.fc2):
            nn.init.xavier_uniform_(fc.weight)
            nn.init.zeros_(fc.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [batch, lookback, n_features]
        Returns: [batch] — predicted annualized vol (positive)
        """
        h1, _       = self.lstm1(x)               # [batch, T, hidden1]
        h1          = self.drop1(h1)
        h2, _       = self.lstm2(h1)               # [batch, T, hidden2]
        last        = self.drop2(h2[:, -1, :])     # [batch, hidden2]
        out         = self.relu(self.fc1(last))    # [batch, 16]
        vol_pred    = self.softplus(self.fc2(out)) # [batch, 1]
        return vol_pred.squeeze(-1)                 # [batch]

# ─Z─ 4. Normalization (fit on train only) ─────────────────
scaler  = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_vl_s  = scaler.transform(X_vl)
X_te_s  = scaler.transform(X_te)
pickle.dump(scaler, open(CONFIG['scaler_path'], 'wb'))



In [18]:
# ============================================================
# SECTION 5: SEQUENCE CONSTRUCTION
# ============================================================

def make_sequences(X: np.ndarray, y: np.ndarray,
                   lookback: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Sliding-window sequence construction.

    Sequence i: features X[i : i+lookback], target y[i+lookback]

    The last observed day in sequence i is index i+lookback-1 (day t).
    The target y[i+lookback] is the realized vol from t+1 to t+H.
    This is causal: no future feature data is included.
    """
    Xs, ys = [], []
    for i in range(len(X) - lookback):
        Xs.append(X[i : i + lookback])
        ys.append(y[i + lookback])
    return (np.array(Xs, dtype=np.float32),
            np.array(ys, dtype=np.float32))


In [19]:
# ── 5. Sequence Construction ─────────────────────────────
LB = CONFIG['lookback']
Xtr_seq, ytr_seq = make_sequences(X_tr_s, y_tr, LB)
Xvl_seq, yvl_seq = make_sequences(X_vl_s, y_vl, LB)
Xte_seq, yte_seq = make_sequences(X_te_s, y_te, LB)
print(f"[SEQUENCES] Train: {Xtr_seq.shape} | "
        f"Val: {Xvl_seq.shape} | Test: {Xte_seq.shape}")

# ── 6. DataLoaders ───────────────────────────────────────
train_ds     = TensorDataset(torch.from_numpy(Xtr_seq),
                                torch.from_numpy(ytr_seq))
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                            shuffle=True, drop_last=True)
X_vl_t = torch.from_numpy(Xvl_seq)
y_vl_t = torch.from_numpy(yvl_seq)
X_te_t = torch.from_numpy(Xte_seq)

[SEQUENCES] Train: (4476, 30, 8) | Val: (1229, 30, 8) | Test: (811, 30, 8)
